# CSC1204 Machine Learning – Easter 2026 Project Examination
## Loan Approval Prediction using Machine Learning
**Program:** BSc Computer Science / BSc Data Science and Analytics | **Year:** 2 | **Semester:** 2  
**Course Code:** CSC1204 | **Examination Date:** April 2026

### **GROUP 5** 
				
SN	Full Name	Reg. Number	Access No.	Program
- 1	SSENKEEZI Christian	S24B23/007	B29658	CS
- 2	KIYINGI Eric	M24B23/019	B27513	CS
- 3	LWANYAGA Ivan	S24B23/023	B29905	CS
- 4	KEMBABAZI Anitah	M24B38/007	B27086	DS
- 5	KIRABO Faith	S24B23/083	B28783	CS
- 6	SSEKUBWA Aslam	S24B23/062	B30119	CS
- 7	NABUUMA Andrea	M24B23/024	B28112	CS
- 8	GRACE Wanja	S24B38/009	B30299	DS
- 9	WALUSIMBI  Amon .C. S24B38/036	B28406	DS
- 10	SSETIMBA Anthony	M24B23/033	B28343	CS

## **Section A: Problem Definition & Dataset Acquisition** 

### (a) Problem Statement

#### Real-World Problem
Uganda's financial institutions lose 5–10% of their annual loan portfolios to defaults. Manual loan-approval processes are slow (taking days), inconsistent, and biased—particularly against women and rural applicants—limiting access to credit for underserved populations.

We propose a supervised binary classification model to predict loan approval (Y/N) based on applicant demographics, financial, and credit profiles. This approach can reduce processing time to seconds, improve consistency, and enable biases in auditing.

#### Why It Matters & Stakeholders
- **Banks & MFIs**: Reduce default rates and streamline operations.
- **Loan Officers**: Gain consistent, auditable decision support.
- **Applicants** (especially rural/female): Access fair, transparent assessments.

#### Decisions Improved by Machine Learning
- Automate approval decisions while reducing bias.
- Better predict default risk for informed lending decisions.
- Enable fairness auditing across demographic groups.

#### Project Objectives
1. Build a binary classification model to predict loan approval.
2. Handle class imbalance using appropriate techniques.
3. Train and compare Random Forest and Logistic Regression models.
4. Perform hyperparameter tuning for both models.
5. Evaluate models using classification metrics (Accuracy, Precision, Recall, F1-score).
6. Analyze feature importance and assess potential bias.

In [ ]:
import pandas as pd

loan_approval = pd.read_csv('Loan_Approval_Data.csv')

loan_approval.info()

### (b) Dataset Acquisition

(i) The dataset was obtained from Kaggle, a very trusted source, titled "Loan Prediction Problem Dataset" available at https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset.

(ii) The dataset contains 614 records (rows), exceeding the minimum of 200. It has 13 features (columns), involving a mix of numeric and categorical variables. The target variable is `Loan_Status`, clearly identifiable as a binary outcome (Y for Approved, N for Rejected), suitable for supervised classification.

(iii) Citation: Analytics Vidhya. (2016). Loan Prediction Problem Dataset. Kaggle. Retrieved from https://www.kaggle.com/datasets/altruistdelhite04/loan-prediction-problem-dataset.

### (c) Data Description

#### (i) Load the Dataset and Display First 10 Rows

In [ ]:
loan_approval.head(10)

#### (ii) Report Dataset Shape, Column Names, and Data Types

In [ ]:
loan_approval.shape

The Dataset has 614 rows and 13 columns

In [ ]:
loan_approval.dtypes

The dataset contains a mix of:
- 5 numeric features (int64, float64)
- 8 categorical features (object)

#### (iii) Data Dictionary - Description of Each Column

| Column | Type | Description |
|--------|------|-------------|
| **Loan_ID** | Identifier | Unique loan identifier — not used in modelling |
| **Gender** | Categorical | Gender of primary applicant (Male / Female) |
| **Married** | Categorical | Marital status (Yes / No) |
| **Dependents** | Categorical | Number of financial dependents (0 / 1 / 2 / 3+) |
| **Education** | Categorical | Education level (Graduate / Not Graduate) |
| **Self_Employed** | Categorical | Employment status (Yes / No) |
| **ApplicantIncome** | Numeric | Monthly income of primary applicant (in thousands) |
| **CoapplicantIncome** | Numeric | Monthly income of co-applicant; 0 if none |
| **LoanAmount** | Numeric | Requested loan amount (in thousands) |
| **Loan_Amount_Term** | Numeric | Repayment tenure in months (e.g., 360 = 30 years) |
| **Credit_History** | Binary | Credit history status (1.0 = Good, 0.0 = Poor) |
| **Property_Area** | Categorical | Property location (Urban / Semiurban / Rural) |
| **Loan_Status** | Target (Binary) | **TARGET:** Loan approval outcome (Y = Approved / N = Rejected) |

#### (iv) Missing Values Analysis

In [ ]:
loan_approval.isnull().sum()

The Dataset contains the bracketed missing values in the Gender(13), Married(3), Dependents(15), Self_Employed(32), LoanAmount(22), Loan_Amount_Term(14), and Credit_History(50) columns


## **SECTION B**

### (a) Summary Statistics & Data Cleaning

#### (i) Descriptive Statistics for Numeric Features

In [ ]:
loan_approval.describe()

**Key Observations:**

- **ApplicantIncome:** Mean (5,403) significantly exceeds median (3,813), indicating right skew. Range 150–81,000 with high std (6,109) reveals extreme outliers. High variability in applicant earning capacity.
- **CoapplicantIncome:** 50th percentile = 1,188.5 but 25th/75th percentiles show concentration near 0. Suggests many applicants lack co-applicants; those with co-applicants have varying additional income (max 41,667).
- **LoanAmount:** 592/614 records present (22 missing). Mean (146.4) > median (128). Range 9–700 with std (85.6) shows wide dispersion. Right-skewed distribution with outlier loans.
- **Loan_Amount_Term:** 600/614 records present (14 missing). Mean (342) pulled down by outliers. 25th/50th/75th percentiles all = 360 months (30 years), showing strong standardization. Minority seek short (12 mo) or long (480 mo) terms.
- **Credit_History:** 564/614 records present (50 missing — highest missingness). Mean (0.84) indicates 84% have good credit (1.0). Highly imbalanced feature; skews dataset toward creditworthy applicants.



In [ ]:
missing_with_pct = pd.DataFrame({
    'Feature': ['Gender', 'Married', 'Dependents', 'Self_Employed', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History'],
    'Missing_Count': [13, 3, 15, 32, 22, 14, 50],
    'Missing_Pct': [13/614*100, 3/614*100, 15/614*100, 32/614*100, 22/614*100, 14/614*100, 50/614*100]
}).round(2)

missing_with_pct

#### (ii) Handle Missing Values

**Missing Values Analysis:**
| Feature | Missing Count | Missing % | Recommendation |
|---------|---------------|-----------|-----------------|
| Self_Employed | 32 | 5.2% | Mode imputation |
| Credit_History | 50 | 8.1% | Mode imputation |
| Gender | 13 | 2.1% | Mode imputation |
| Dependents | 15 | 2.4% | Mode imputation |
| LoanAmount | 22 | 3.6% | Median imputation |
| Loan_Amount_Term | 14 | 2.3% | Mode imputation |
| Married | 3 | 0.5% | Mode imputation |

**Strategy Justification:**

1. **Categorical Features (Gender, Married, Dependents, Self_Employed, Credit_History):**
   - **Strategy:** Mode imputation (most frequent value)
   - **Rationale:** Missing rates are low (<9%). Mode preserves the original distribution without introducing bias. For binary/nominal features, this is the standard approach.

2. **LoanAmount (22 missing, 3.6%):**
   - **Strategy:** Median imputation
   - **Rationale:** Right-skewed distribution (mean 146.4 > median 128; std 85.6). Median is more robust to outliers than mean and prevents inflation of loan values. Low missing rate (<4%) permits imputation.

3. **Loan_Amount_Term (14 missing, 2.3%):**
   - **Strategy:** Mode imputation
   - **Rationale:** Heavily concentrated at 360 months (25th/50th/75th percentiles all = 360). Mode (360) represents the standard loan term and preserves this concentration. More appropriate than median.

**Overall Justification:**
- All missing rates < 10% are suitable for imputation without risk of information loss.
- No features deleted; all 614 rows retained for maximum dataset size.
- Imputation methods match feature characteristics (categorical → mode; skewed numeric → median; concentrated numeric → mode).
- Preserves data distributions and avoids introducing artificial patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

#we make a copy of the original dataset to work with, so that we can always refer back to the raw data if needed
loan_approval_copy = loan_approval.copy()

In [ ]:
# Work on a clean copy (drop Loan_ID — it is a non-predictive identifier)
loan_cleaned = loan_approval_copy.drop(columns=['Loan_ID'])

# Fill categorical features with mode
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']

for col in categorical_cols:
    loan_cleaned[col] = loan_cleaned[col].fillna(loan_cleaned[col].mode()[0])

# Fill numeric features
loan_cleaned['LoanAmount'] = loan_cleaned['LoanAmount'].fillna(loan_cleaned['LoanAmount'].median())

loan_cleaned['Loan_Amount_Term'] = loan_cleaned['Loan_Amount_Term'].fillna(loan_cleaned['Loan_Amount_Term'].mode()[0])

# Confirm no missing values
loan_cleaned.isnull().sum()

#### (iii) Identify and Handle Outliers

**Approach:**
- **Method:** Interquartile Range (IQR) method
- **Definition:** Outliers are values beyond Q1 - 1.5×IQR and Q3 + 1.5×IQR
- **Handling:** Cap outliers using IQR bounds (Q1 − 1.5×IQR, Q3 + 1.5×IQR)
- **Rationale:** Income and loan amount can legitimately have high values; capping preserves these while controlling extreme outliers

In [ ]:
#detecting and visualizing outliers in numeric features

import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(12, 6)) 

numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(1, 3, i)  # 1 row, 3 columns
    sns.boxplot(y=loan_cleaned[col])
    plt.title(col)

plt.tight_layout()
plt.show()

The dataset has outliers 

Although Loan_Amount_Term is a numeric feature, it was excluded from outlier treatment because it represents discrete loan durations with a strong concentration at 360 months. 

Applying IQR-based outlier detection would incorrectly classify valid loan terms (such as 120 or 480 months) as outliers. Therefore, it was retained without modification to preserve meaningful financial structure.

In [ ]:
for col in ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']:
    Q1 = loan_cleaned[col].quantile(0.25)
    Q3 = loan_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    loan_cleaned[col] = np.clip(loan_cleaned[col], lower_bound, upper_bound)

In [ ]:
plt.figure(figsize=(12, 6))

for i, col in enumerate(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount'], 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=loan_cleaned[col])
    plt.title(f'{col} (After Capping)')

plt.tight_layout()
plt.show()

Outliers were identified using boxplots for key numeric features: ApplicantIncome, CoapplicantIncome, and LoanAmount. These features showed right-skewed distributions with extreme high values.

To handle outliers, the Interquartile Range (IQR) method was applied. Values below Q1 − 1.5×IQR and above Q3 + 1.5×IQR were capped at the respective bounds using clipping.

This approach was chosen because:
- It reduces the influence of extreme values without removing data points
- It preserves dataset size (important for small datasets)
- It is robust for skewed financial data

#### (iv) Encode Categorical Variables

**Encoding Strategy:**

All categorical variables are encoded using **one-hot encoding** to convert them into numerical format suitable for machine learning models.

- One-hot encoding creates binary columns for each category in a feature.
- It was chosen because the dataset contains **nominal categorical variables** (e.g., Gender, Property_Area) with no inherent order.
- Label encoding was avoided as it can introduce unintended ordinal relationships between categories.

To prevent multicollinearity, the parameter `drop_first=True` was applied, which removes one redundant category from each feature.

**Target Variable Encoding:**
The target variable `Loan_Status` is encoded as:
- 1 → Approved (Y)  
- 0 → Rejected (N)  

This binary representation is standard for classification tasks.

In [ ]:
# Encode categorical variables
loan_encoded = pd.get_dummies(loan_cleaned, drop_first=True)

# Encode target variable
loan_encoded['Loan_Status'] = loan_encoded['Loan_Status_Y']
loan_encoded.drop(columns=['Loan_Status_Y'], inplace=True)
loan_encoded['Loan_Status'] = loan_encoded['Loan_Status'].astype(int)

# View result
loan_encoded.head()

In [ ]:
loan_encoded['Loan_Status'].value_counts()

#### (b) Univariate Analysis

In [ ]:
#Target Variable Distribution
plt.figure(figsize=(6,4))
sns.countplot(x=loan_encoded['Loan_Status'])
plt.title('Distribution of Loan Status')
plt.xlabel('Loan Status (0 = Rejected, 1 = Approved)')
plt.ylabel('Count')
plt.show()

The target variable shows a class imbalance, with a higher number of approved loans (1) compared to rejected loans (0). This indicates that the dataset is skewed towards approval cases, which may affect model performance and requires consideration during model evaluation.

In [ ]:
#histograms for numeric features
numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

plt.figure(figsize=(12,6))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(1,3,i)
    sns.histplot(loan_encoded[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()

The numeric features show right-skewed distributions, particularly for ApplicantIncome and CoapplicantIncome, indicating the presence of high-income outliers. LoanAmount is also slightly skewed, though less extreme after outlier handling.

### (c) Bivariate & Multivariate Analysis

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))

corr_matrix = loan_encoded.corr()

sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)

plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

corr_with_target = loan_encoded.corr()[['Loan_Status']].sort_values(by='Loan_Status', ascending=False)

sns.heatmap(corr_with_target, annot=True, cmap='coolwarm')

plt.title('Correlation with Target Variable (Loan_Status)')
plt.show()

Found this clearer for better understanding since the previous heatmat looked crowded 

In [ ]:
loan_encoded.corr()['Loan_Status'].sort_values(ascending=False)

The correlation heatmap reveals the relationships between features and the target variable (Loan_Status), as well as relationships among the features themselves.

**Key Findings:**

- **Credit_History (0.54)** shows the strongest positive correlation with Loan_Status, making it the most influential predictor of loan approval. Applicants with a good credit history are significantly more likely to be approved.

- **Property_Area_Semiurban (0.14)** has a weak positive correlation, suggesting that applicants in semi-urban areas have slightly higher chances of loan approval compared to other areas.

- **Married_Yes (0.09)** and **Dependents_2 (0.06)** show very weak positive correlations, indicating minimal influence on loan approval.

- Most other features, including **ApplicantIncome**, **CoapplicantIncome**, and **LoanAmount**, have correlations close to zero, suggesting that they do not have a strong linear relationship with the target variable.

- Some features show weak negative correlations, such as:
  - **Education_Not Graduate (-0.086)**, indicating that non-graduates are slightly less likely to be approved.
  - **LoanAmount (-0.047)** and **Property_Area_Urban (-0.044)**, though their effects are minimal.

**Feature Relationships:**

- There is a moderate positive correlation between **ApplicantIncome** and **LoanAmount**, indicating that higher-income applicants tend to request larger loans.

- Overall, most features show low correlation with each other, suggesting **low multicollinearity**, which is beneficial for model stability.

**Conclusion:**

Credit history stands out as the dominant factor influencing loan approval, while most other features have relatively weak individual relationships with the target. This suggests that the model will rely heavily on Credit_History, with other features contributing marginally.

#### (ii) Bivariate Analysis: Grouped Visualizations

To explore the relationships between key features and the target variable (Loan_Status), grouped visualizations are used.

Since the target variable is binary (0 = Rejected, 1 = Approved), different types of plots are selected based on the nature of the features:

- A **countplot** is used for Credit_History because it is a binary categorical variable, allowing clear comparison of counts across loan approval classes.
- A **boxplot** is used for LoanAmount because it is a numeric feature, allowing comparison of distributions between approved and rejected applicants.

Two important features are selected based on earlier correlation analysis:
- Credit_History (strongest predictor)
- LoanAmount (financial feature with weak correlation)

In [ ]:
sns.countplot(x='Credit_History', hue='Loan_Status', data=loan_encoded)
plt.title('Credit History vs Loan Status')
plt.show()

In [ ]:
pd.crosstab(loan_encoded['Loan_Status'], loan_encoded['Credit_History'])

**Credit History vs Loan Status:**

The relationship between credit history and loan approval is very strong. From the cross-tabulation, the majority of approved applicants (415 out of 422) have a credit history of 1 (good), while only a very small number (7) with poor credit history were approved.

For rejected applicants, a larger proportion have poor credit history (82), although some applicants with good credit history (110) were still rejected.

In the boxplot, the approved group appears as a flat line at 1 because Credit_History is a binary variable and most approved applicants share the same value. This results in no variation within that group.

Overall, this confirms that credit history is the most influential factor in loan approval, although it is not the only determinant, as some applicants with good credit are still rejected.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=loan_encoded['Loan_Status'], y=loan_encoded['LoanAmount'])
plt.title('Loan Amount vs Loan Status')
plt.xlabel('Loan Status (0 = Rejected, 1 = Approved)')
plt.ylabel('Loan Amount')
plt.show()

**Loan Amount vs Loan Status:**

The boxplot shows that the distribution of loan amounts is very similar for both approved and rejected applicants. The median values and overall spread overlap significantly between the two groups.

Although there are some variations and a few extreme values, there is no clear separation between approved and rejected loans based on loan amount alone.

This indicates that loan amount is not a strong predictor of loan approval, which is consistent with the earlier correlation analysis showing a weak relationship with the target variable.

#### (iii) Pairplot of Key Features

A pairplot is generated to visualize pairwise relationships between selected important features and the target variable.

The features included are ApplicantIncome, LoanAmount, Credit_History, and Loan_Status, as they are among the most relevant predictors identified during earlier analysis.




In [ ]:
selected_features = ['ApplicantIncome', 'LoanAmount', 'Credit_History', 'Loan_Status']

sns.pairplot(loan_encoded[selected_features], hue='Loan_Status')

plt.show()


**Observations:**

- Credit_History shows a clear separation between approved and rejected loans, confirming its strong influence on loan approval.
- ApplicantIncome and LoanAmount do not show clear separation between classes, indicating weaker predictive power.
- There is a slight positive relationship between ApplicantIncome and LoanAmount, suggesting that higher-income applicants tend to request larger loans.
- Most feature pairs show overlapping distributions between classes, indicating that individual features alone may not fully separate approved and rejected cases.

**Hence:**

The pairplot reinforces earlier findings that Credit_History is the most significant predictor, while other features contribute less distinctly and may require combination through machine learning models for effective prediction.

#### (d) Key Insights from EDA

- **Credit_History is the strongest predictor** of loan approval, showing a clear positive relationship with the target variable. This feature is expected to play a major role in model performance.

- The dataset exhibits **class imbalance**, with more approved (1) than rejected (0) loans. This may affect model evaluation and will require careful use of evaluation metrics such as precision, recall, and F1-score.

- **ApplicantIncome and LoanAmount are right-skewed** and contained outliers, which were handled using IQR capping. Despite this, they show weak direct correlation with the target variable.

- Most features show **low correlation with each other**, indicating minimal multicollinearity and suggesting that models are less likely to be negatively affected by redundant features.

- **Categorical variables required encoding**, and one-hot encoding was applied to convert them into a machine-learning-friendly format without introducing artificial ordering.

#### (ii) Important Features for Modelling

Based on the exploratory data analysis, the following features are expected to be most important for the model:

- **Credit_History:** This is the most significant feature, showing the strongest positive correlation with Loan_Status. Both the correlation analysis and grouped visualizations confirm that applicants with good credit history are far more likely to be approved.

- **Property_Area_Semiurban:** This feature shows a moderate positive relationship with loan approval, suggesting that applicants from semi-urban areas may have a higher likelihood of approval compared to other regions.

- **Married_Yes:** This feature has a slight positive correlation with the target variable, indicating that marital status may have a minor influence on loan approval decisions.

- **ApplicantIncome and LoanAmount:** Although these features show weak direct correlation with the target, they represent important financial characteristics of applicants. They may still contribute to the model when combined with other features.

**Conclusion:**

Credit_History is expected to be the dominant predictor, while other features such as property area, marital status, and financial variables are likely to provide additional supporting information for the model.

## **SECTION C**

### (a) Data Preparation

#### (i) Feature Matrix (X) and Target Vector (y)

To prepare the data for modelling, the dataset is divided into:

- **Feature Matrix (X):** Contains all independent variables used to predict loan approval.
- **Target Vector (y):** Contains the dependent variable, which is Loan_Status.

The target variable `Loan_Status` represents whether a loan is approved (1) or rejected (0). All other features are used as input variables for the model.

The dataset has already been cleaned, encoded, and transformed into a fully numeric format, making it suitable for machine learning algorithms.

In [ ]:
# Define features and target
X = loan_encoded.drop(columns=['Loan_Status'])
y = loan_encoded['Loan_Status']

# Check shapes
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

#### (ii) Train-Test Split

The dataset is split into training and testing sets to evaluate model performance on unseen data.

- **Training Set (80%)** is used to train the machine learning models.
- **Testing Set (20%)** is used to evaluate how well the trained models generalize to new data.

A random state of 42 is used to ensure reproducibility of results.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

The dataset was successfully split into training and testing sets using an 80/20 ratio. Stratified sampling was applied to preserve the class distribution of the target variable in both sets.

This ensures that both training and testing data are representative of the original dataset, which is important for reliable model evaluation.


#### (iii) Feature Scaling

Feature scaling was applied to ensure that numeric features are on a similar scale, which is important for certain machine learning algorithms.

Scaling is particularly important for **Logistic Regression**, as it is sensitive to the magnitude of feature values. Features like ApplicantIncome and LoanAmount have large ranges and could dominate the model if not scaled.

On the other hand, **Random Forest** is a tree-based model and does not require feature scaling, as it splits data based on feature thresholds rather than distances.

Standardization (StandardScaler) was used to transform the features so that they have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

The training data was scaled using StandardScaler, and the same transformation was applied to the test data. 

This ensures that the model is trained on standardized features while avoiding data leakage, since the scaler is fit only on the training set.

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

The scaled data is converted back into a DataFrame to retain the original feature names. This improves readability, simplifies interpretation of results, and makes it easier to analyze feature importance in later stages of the project.

### (b) Model 1: Primary Model

#### (i) Algorithm Selection and Justification

The primary model selected for this task is **Random Forest Classifier**, a supervised machine learning algorithm used for classification problems.

Random Forest is an ensemble learning method that builds multiple decision trees and combines their predictions to improve overall accuracy and reduce overfitting.

**Justification:**

- **Handles Non-linear Relationships:** Random Forest can capture complex patterns in the data, unlike linear models.
- **Robust to Outliers:** It performs well even when the dataset contains outliers, which were observed in income and loan amount features.
- **Feature Importance:** It provides feature importance scores, which are useful for interpreting which variables influence loan approval.
- **Works Well with Mixed Data:** It can handle both numerical and encoded categorical features effectively.
- **Less Sensitive to Scaling:** Unlike some algorithms, Random Forest does not require feature scaling, making it robust to different feature ranges.

Given these advantages and the nature of the dataset, Random Forest is well-suited as the primary model for predicting loan approval.

#### (ii) Model Training

The Random Forest Classifier is trained using the training dataset. The model learns patterns and relationships between the input features and the target variable (Loan_Status) during this phase.

The trained model will later be evaluated on unseen test data to assess its performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train) ##Since Random Forest does NOT require scaling, use X_train instead of X_train_scaled

#### (iii) Hyperparameter Tuning

To improve model performance, hyperparameter tuning was performed on the Random Forest Classifier by testing different configurations of key parameters.

The following hyperparameters were varied:
- **n_estimators:** Number of trees in the forest
- **max_depth:** Maximum depth of each tree
- **min_samples_split:** Minimum number of samples required to split a node

Model performance for each configuration was evaluated using **F1-score**, which is appropriate for imbalanced classification problems.

In [ ]:
from sklearn.metrics import f1_score

# Store results
results = []

# Configuration 1
rf1 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf1.fit(X_train, y_train)
y_pred1 = rf1.predict(X_test)
f1_1 = f1_score(y_test, y_pred1)
results.append(("Config 1", f1_1))

# Configuration 2
rf2 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf2.fit(X_train, y_train)
y_pred2 = rf2.predict(X_test)
f1_2 = f1_score(y_test, y_pred2)
results.append(("Config 2", f1_2))

# Configuration 3
rf3 = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42)
rf3.fit(X_train, y_train)
y_pred3 = rf3.predict(X_test)
f1_3 = f1_score(y_test, y_pred3)
results.append(("Config 3", f1_3))

# Display results
for config, score in results:
    print(f"{config}: F1-score = {score:.4f}")

The performance of the three configurations shows that:

- **Configuration 1** achieved the highest F1-score (0.9032), making it the best-performing model.
- Increasing the number of trees and depth in Configurations 2 and 3 did not improve performance, suggesting that a simpler model generalizes better for this dataset.
- More complex models may introduce slight overfitting without significant gains in predictive accuracy.

Based on these results, **Configuration 1 is selected as the optimal Random Forest model** for further evaluation.

#### (iv) Hyperparameter Explanation

The following hyperparameters were tuned for the Random Forest model:

- **n_estimators:**  
  This represents the number of decision trees in the forest. Increasing the number of trees generally improves model performance by reducing variance and making predictions more stable. However, too many trees can increase computational cost without significant performance gains.

- **max_depth:**  
  This controls the maximum depth of each decision tree. A smaller depth (e.g., 5) prevents the model from becoming too complex and helps reduce overfitting. A larger depth allows the model to capture more detailed patterns but may lead to overfitting.

- **min_samples_split:**  
  This defines the minimum number of samples required to split an internal node. Higher values make the model more conservative by reducing the number of splits, which can help prevent overfitting. Lower values allow the model to learn more detailed patterns but may increase complexity.

**Overall Effect:**

These hyperparameters control the balance between **bias and variance**. Proper tuning ensures that the model is neither too simple (underfitting) nor too complex (overfitting), resulting in better generalization to unseen data.

### (c) Model 2: Comparison Model

#### (i) Algorithm Selection and Justification

The second model selected is **Logistic Regression**, a supervised learning algorithm used for binary classification tasks.

Logistic Regression models the probability of a binary outcome using a linear relationship between the input features and the target variable.

**Justification:**

- **Baseline Model:** It provides a simple and interpretable baseline to compare against the more complex Random Forest model.
- **Efficient and Fast:** It is computationally efficient and works well on smaller datasets.
- **Interpretable:** The model coefficients help explain how each feature influences loan approval.
- **Works Well with Scaled Data:** Since feature scaling was applied, Logistic Regression can perform optimally on this dataset.

This model is chosen to evaluate whether a simpler linear model can achieve comparable performance to a more complex ensemble method.

#### (ii) Model Training

The Logistic Regression model is trained using the scaled training dataset. Scaling is important for this algorithm because it is sensitive to the magnitude of feature values.

The trained model will later be evaluated and compared with the Random Forest model.

In [ ]:
# Importing Logistic Regression
from sklearn.linear_model import LogisticRegression

# Initializing model
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# Training model on scaled data
lr_model.fit(X_train_scaled, y_train)

#### (iii) Hyperparameter Tuning

Hyperparameter tuning was performed on the Logistic Regression model by testing different configurations of the regularization parameter.

The following hyperparameters were varied:
- **C:** Inverse of regularization strength (controls how much the model is penalized for complexity)
- **solver:** Algorithm used for optimization

Model performance was evaluated using **F1-score**, which is suitable for imbalanced classification problems.

In [ ]:
from sklearn.metrics import f1_score

lr_results = []

# Configuration 1
lr1 = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
lr1.fit(X_train_scaled, y_train)
y_pred_lr1 = lr1.predict(X_test_scaled)
f1_lr1 = f1_score(y_test, y_pred_lr1)
lr_results.append(("Config 1", f1_lr1))

# Configuration 2
lr2 = LogisticRegression(C=0.1, solver='liblinear', max_iter=1000, random_state=42)
lr2.fit(X_train_scaled, y_train)
y_pred_lr2 = lr2.predict(X_test_scaled)
f1_lr2 = f1_score(y_test, y_pred_lr2)
lr_results.append(("Config 2", f1_lr2))

# Displaying results
for config, score in lr_results:
    print(f"{config}: F1-score = {score:.4f}")

The performance results show that both configurations achieved the same F1-score of 0.9032.

This indicates that changing the regularization strength (C) and solver did not significantly impact the model's performance on this dataset.

Since both configurations perform equally well, **Configuration 1 is selected** as the preferred model due to its default settings and computational simplicity.

This result suggests that the dataset is relatively stable and not highly sensitive to changes in Logistic Regression hyperparameters.

## **SECTION D**

### (a) Evaluation Metrics

Since this is a **classification task**, the following evaluation metrics are used:

- **Accuracy:** Measures overall correctness of the model
- **Precision:** Measures how many predicted positives are actually correct
- **Recall:** Measures how many actual positives are correctly identified
- **F1-Score:** Harmonic mean of precision and recall, suitable for imbalanced datasets
- **Confusion Matrix:** Shows the distribution of correct and incorrect predictions

The best-performing configuration of each model is evaluated using these metrics.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Random Forest predictions
y_pred_rf = rf_model.predict(X_test)

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Classification Report
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

**Random Forest Model Evaluation:**

The Random Forest model achieved an overall accuracy of **84%**, indicating good predictive performance on the test dataset.

- For the **approved class (1)**:
  - Precision: 0.85
  - Recall: 0.93
  - F1-score: 0.89  
  This shows that the model performs very well in identifying approved loans, correctly capturing most positive cases.

- For the **rejected class (0)**:
  - Precision: 0.80
  - Recall: 0.63
  - F1-score: 0.71  
  The model is less effective at identifying rejected applications, as seen from the lower recall. This suggests that some rejected cases are being misclassified as approved.

- The **macro average F1-score (0.80)** indicates moderate balance between the two classes, while the **weighted average F1-score (0.83)** reflects stronger performance on the majority class.

**Conclusion:**

The Random Forest model performs well overall, particularly in predicting loan approvals. However, it struggles somewhat with correctly identifying rejected applications, likely due to class imbalance in the dataset.

In [ ]:
# Logistic Regression predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6,4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens')
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Classification Report
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

**Logistic Regression Model Evaluation:**

The Logistic Regression model achieved an overall accuracy of **85%**, slightly higher than the Random Forest model.

- For the **approved class (1)**:
  - Precision: 0.83
  - Recall: 0.99
  - F1-score: 0.90  
  The model performs extremely well in identifying approved loans, correctly capturing almost all positive cases.

- For the **rejected class (0)**:
  - Precision: 0.95
  - Recall: 0.55
  - F1-score: 0.70  
  Although precision is very high, recall is relatively low, meaning that many rejected applications are incorrectly classified as approved.

- The **macro average F1-score (0.80)** indicates moderate balance between classes, while the **weighted average F1-score (0.84)** reflects strong overall performance.

**Conclusion:**

The Logistic Regression model performs slightly better overall than Random Forest in terms of accuracy and F1-score. However, it has a significant limitation in identifying rejected applications due to low recall for class 0.

#### (b) Model Comparison

(i) Comparison Table

The table below summarizes the performance of the best configurations of both models using key evaluation metrics.

In [ ]:

comparison_table = pd.DataFrame({
    "Model": ["Random Forest (Best)", "Logistic Regression (Best)"],
    "Accuracy": [0.84, 0.85],
    "Precision (Class 1)": [0.85, 0.83],
    "Recall (Class 1)": [0.93, 0.99],
    "F1-Score (Class 1)": [0.89, 0.90],
    "Precision (Class 0)": [0.80, 0.95],
    "Recall (Class 0)": [0.63, 0.55],
    "F1-Score (Class 0)": [0.71, 0.70]
})

comparison_table

#### (ii) Best Model Selection

Both Random Forest and Logistic Regression achieved very similar overall performance, with Logistic Regression slightly higher in accuracy (0.85 vs 0.84) and F1-score for the approved class.

However, a closer analysis shows important differences:

- Logistic Regression has extremely high recall for approved loans (0.99), meaning it correctly identifies almost all approved applications.
- Random Forest provides better recall for rejected loans (0.63 vs 0.55), making it more balanced in detecting both classes.

**Final Decision:**

The **Random Forest model is selected as the best-performing model**.

**Justification:**

Although Logistic Regression has slightly higher accuracy, Random Forest offers a better balance between detecting approved and rejected applications. This is important in real-world lending, where failing to identify risky (rejected) applicants can lead to financial loss.

Therefore, Random Forest is preferred due to its more balanced and reliable performance across both classes.

#### (iii) Trade-offs Between Models

Both Random Forest and Logistic Regression perform well, but they differ in several important ways:

- **Accuracy vs Balance:**  
  Logistic Regression achieves slightly higher accuracy and F1-score for approved loans, but Random Forest provides better balance by performing relatively better on rejected cases.

- **Recall for Approved Loans (Class 1):**  
  Logistic Regression has very high recall (0.99), meaning it rarely misses approved applicants. However, this comes at the cost of misclassifying some rejected applicants as approved.

- **Recall for Rejected Loans (Class 0):**  
  Random Forest performs better in identifying rejected applications (recall = 0.63 vs 0.55), making it more reliable for detecting risky applicants.

- **Model Complexity:**  
  Logistic Regression is simpler, faster, and easier to interpret, making it suitable for baseline models and explainability.  
  Random Forest is more complex but captures non-linear relationships more effectively.

- **Risk Consideration:**  
  Logistic Regression tends to favor approving loans, which may increase financial risk.  
  Random Forest is more conservative and better at identifying potential defaulters.

**Conclusion:**

The choice between the two models depends on the objective. If the goal is to maximize approvals, Logistic Regression is suitable. However, if the goal is to balance approvals with risk control, Random Forest is the better option.

In [ ]:
##Visualization 1: Confusion Matrix (Random Forest)

plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

**Random Forest Confusion Matrix Interpretation:**

The confusion matrix shows that the model correctly predicts a large number of approved loans (true positives) and a reasonable number of rejected loans (true negatives).

However, there are some misclassifications where rejected applications are predicted as approved, indicating difficulty in identifying all risky applicants.

Overall, the model demonstrates strong performance, especially for the majority class (approved loans).

In [ ]:
##Visualization 2: Confusion Matrix (Logistic Regression)

plt.figure(figsize=(6,4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens')
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

**Logistic Regression Confusion Matrix Interpretation:**

The confusion matrix shows that the model performs extremely well in predicting approved loans, with very few false negatives.

However, it misclassifies a higher number of rejected applications as approved, indicating weaker performance in identifying risky applicants.

This confirms that the model is biased toward predicting approvals, which may increase financial risk.

In [ ]:
##Visualization 3: Feature Importance 

# Get feature importances
importances = rf_model.feature_importances_
features = X.columns

# Create DataFrame
feat_imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(10,6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'])
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.gca().invert_yaxis()
plt.show()


**Feature Importance Interpretation:**

The feature importance plot shows that **Credit_History** is the most influential feature in predicting loan approval, with a significantly higher importance score compared to all other variables.

This indicates that the model relies heavily on an applicant’s credit history when making approval decisions.

Other features such as **ApplicantIncome, LoanAmount, CoapplicantIncome, and Loan_Amount_Term** contribute moderately to the model, suggesting they provide additional but less dominant predictive value.

Several features have relatively low importance, indicating minimal influence on the model’s predictions.

These findings are consistent with earlier EDA results, where Credit_History showed the strongest correlation with the target variable. This confirms that credit history is the key driver of loan approval decisions in this dataset.